In [2]:
# ==========================================================
# Step 1: Import Required Libraries
# ==========================================================

import pandas as pd
import numpy as np
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# ==========================================================
# Step 2: Load Saved Model and Files
# ==========================================================

model = joblib.load("xgboost_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")
feature_columns = joblib.load("feature_columns.pkl")

print("Model loaded successfully!")
print("Label Encoder loaded successfully!")
print("Feature Columns loaded successfully!")
print("Total Features:", len(feature_columns))

Model loaded successfully!
Label Encoder loaded successfully!
Feature Columns loaded successfully!
Total Features: 131


In [4]:
# ==========================================================
# Step 3: Prediction Function
# ==========================================================

def predict_disease(selected_symptoms):

    # Create feature vector
    input_data = np.zeros(len(feature_columns))

    for symptom in selected_symptoms:
        if symptom in feature_columns:
            index = feature_columns.index(symptom)
            input_data[index] = 1

    input_data = input_data.reshape(1, -1)

    # Prediction
    prediction = model.predict(input_data)

    disease = label_encoder.inverse_transform(prediction)[0]

    # Confidence Score
    probabilities = model.predict_proba(input_data)

    confidence = np.max(probabilities) * 100

    return disease, confidence

In [7]:
sample_symptoms = [
    "itching",
    "skin_rash",
    "nodal_skin_eruptions"
]

input_data = np.zeros(len(feature_columns))

for symptom in sample_symptoms:
    if symptom in feature_columns:
        index = feature_columns.index(symptom)
        input_data[index] = 1

input_data = input_data.reshape(1, -1)

probs = model.predict_proba(input_data)

print("Shape:", probs.shape)
print("Sum:", probs.sum())
print("Maximum Probability:", probs.max())
print("First 10 Probabilities:")
print(probs[0][:10])

Shape: (1, 41)
Sum: 1.0
Maximum Probability: 0.05167336
First 10 Probabilities:
[0.02680276 0.02900843 0.02477645 0.02700554 0.02368155 0.01933239
 0.01856134 0.02107297 0.02076203 0.02208556]


In [14]:
model = joblib.load("xgboost_model.pkl")

print(model.get_params()["objective"])

multi:softprob


In [15]:
print("Number of Trees:", model.n_estimators)
print("Best Iteration:", getattr(model, "best_iteration", None))

Number of Trees: None
Best Iteration: None


In [16]:
booster = model.get_booster()
print("Number of Boosted Rounds:", booster.num_boosted_rounds())

Number of Boosted Rounds: 100


In [17]:
# ==========================================================
# Model Verification Test
# ==========================================================

import numpy as np
import joblib

# Load saved files
model = joblib.load("xgboost_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")
feature_columns = joblib.load("feature_columns.pkl")

# ----------------------------------------------------------
# Test Symptoms
# ----------------------------------------------------------

test_symptoms = [
    "itching",
    "skin_rash",
    "nodal_skin_eruptions"
]

# ----------------------------------------------------------
# Create Feature Vector
# ----------------------------------------------------------

input_data = np.zeros(len(feature_columns))

for symptom in test_symptoms:
    if symptom in feature_columns:
        index = feature_columns.index(symptom)
        input_data[index] = 1

input_data = input_data.reshape(1, -1)

# ----------------------------------------------------------
# Predict Disease
# ----------------------------------------------------------

prediction = model.predict(input_data)

predicted_disease = label_encoder.inverse_transform(prediction)[0]

print("=" * 50)
print("MODEL VERIFICATION")
print("=" * 50)

print("Selected Symptoms:")
print(test_symptoms)

print("\nPredicted Disease:")
print(predicted_disease)

# ----------------------------------------------------------
# Prediction Probability
# ----------------------------------------------------------

probabilities = model.predict_proba(input_data)[0]

predicted_index = prediction[0]

print("\nProbability of Predicted Disease:")
print(f"{probabilities[predicted_index]*100:.2f}%")

# ----------------------------------------------------------
# Top 5 Predictions
# ----------------------------------------------------------

print("\nTop 5 Predictions")
print("-" * 50)

top5 = np.argsort(probabilities)[::-1][:5]

for i in top5:
    disease = label_encoder.inverse_transform([i])[0]
    print(f"{disease:<40} {probabilities[i]*100:.2f}%")

print("=" * 50)

MODEL VERIFICATION
Selected Symptoms:
['itching', 'skin_rash', 'nodal_skin_eruptions']

Predicted Disease:
Fungal infection

Probability of Predicted Disease:
5.17%

Top 5 Predictions
--------------------------------------------------
Fungal infection                         5.17%
Jaundice                                 4.28%
Drug Reaction                            3.66%
AIDS                                     2.90%
Tuberculosis                             2.81%
